In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# Notebook location:
# AthletiQ/ai-engine/notebooks/AthletiQ_Performance_Analysis.ipynb

NOTEBOOK_DIR = Path.cwd().resolve()

# Find the AthletiQ project root even if Jupyter starts in another directory.
candidates = [
    NOTEBOOK_DIR,
    *NOTEBOOK_DIR.parents,
]

PROJECT_ROOT = None

for path in candidates:
    if (path / "outputs/dsa/football2_unified_player_features.csv").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the AthletiQ project root.\n"
        "Expected: outputs/dsa/football2_unified_player_features.csv\n"
        f"Notebook working directory: {NOTEBOOK_DIR}"
    )

FEATURE_PATH = (
    PROJECT_ROOT
    / "outputs/dsa/football2_unified_player_features.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs/performance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TEMPLATE_PATH = (
    OUTPUT_DIR / "player_performance_labels_template.csv"
)

print("Notebook working directory:", NOTEBOOK_DIR)
print("AthletiQ project root:", PROJECT_ROOT)
print("Feature dataset exists:", FEATURE_PATH.exists())
print("Label template exists:", LABEL_TEMPLATE_PATH.exists())
print("Output directory:", OUTPUT_DIR)

Notebook working directory: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks
AthletiQ project root: /mnt/d/My Projects/AthletiQ/AthletiQ
Feature dataset exists: True
Label template exists: True
Output directory: /mnt/d/My Projects/AthletiQ/AthletiQ/outputs/performance


In [2]:
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Feature dataset not found: {FEATURE_PATH}")

features = pd.read_csv(FEATURE_PATH)

labels = (
    pd.read_csv(LABEL_TEMPLATE_PATH)
    if LABEL_TEMPLATE_PATH.exists()
    else None
)

print("========== DATASET OVERVIEW ==========")
print(f"Rows: {features.shape[0]}")
print(f"Columns: {features.shape[1]}")
print(f"Numeric columns: {features.select_dtypes(include=np.number).shape[1]}")
print(f"Duplicate rows: {features.duplicated().sum()}")

if "canonical_id" in features.columns:
    print(
        "Duplicate canonical IDs:",
        features["canonical_id"].duplicated().sum()
    )

display(features.head(10))

if labels is not None:
    print("\n========== LABEL TEMPLATE ==========")
    print("Shape:", labels.shape)
    display(labels.head())
else:
    print("\nLabel template not found.")

========== DATASET OVERVIEW ==========
Rows: 76
Columns: 46
Numeric columns: 44
Duplicate rows: 0
Duplicate canonical IDs: 0


,canonical_id,movement_first_frame,movement_last_frame,movement_observed_rows,movement_track_span_seconds,movement_continuous_segments,movement_coverage_ratio,movement_valid_movement_steps,movement_excluded_outlier_steps,movement_distance_pixels,movement_valid_movement_time_seconds,movement_average_speed_pixels_per_second,movement_maximum_step_speed_pixels_per_second,movement_median_step_speed_pixels_per_second,movement_maximum_positive_acceleration_pixels_per_second_squared,movement_maximum_deceleration_magnitude_pixels_per_second_squared,movement_experimental_sprint_threshold_pixels_per_second,movement_experimental_sprint_count,movement_experimental_sprint_distance_pixels,movement_experimental_sprint_duration_seconds,spatial_zone_row,spatial_zone_col,spatial_observed_frames,spatial_first_frame,spatial_last_frame,spatial_mean_x_px,spatial_mean_y_px,spatial_observed_seconds,possession_estimated_possession_frames,possession_estimated_possession_seconds,possession_mean_ball_distance_px,possession_mean_heuristic_confidence,kdtree_observed_frames,kdtree_mean_nearest_distance_px,kdtree_median_nearest_distance_px,kdtree_mean_neighbors_within_radius,graph_degree,graph_weighted_degree,graph_same_team_edges,graph_opponent_edges,graph_component_id,graph_component_size,baseline_activity_index,score_type,tracking_reliability_index,activity_score_status
0,1,0.0,287.0,288.0,11.48,1.0,1.000000,263.0,24.0,306.716016,10.52,29.155515,175.587500,21.980677,864.260108,881.903510,32.469430,1.0,98.945311,1.24,1.25,1.25,288,153.500000,226.250000,278.214488,251.809381,2.880000,NaN,NaN,NaN,NaN,288,69.697477,62.851971,0.906250,8,261.0,2,6,1,18,45.64,within_video_activity_percentile_not_ml,100.00,OK
1,2,0.0,170.0,171.0,6.80,1.0,1.000000,170.0,0.0,249.964635,6.80,36.759505,110.691376,36.486707,473.335521,715.861952,43.858238,1.0,63.352893,0.88,2.00,3.50,171,42.500000,127.000000,630.784384,369.578259,3.420000,NaN,NaN,NaN,NaN,171,73.458165,73.084400,0.549708,1,93.0,1,0,1,18,42.77,within_video_activity_percentile_not_ml,100.00,OK
2,3,0.0,155.0,155.0,6.20,1.0,0.993590,154.0,0.0,195.161761,6.20,31.477703,136.087104,25.314318,608.858378,752.002032,34.872536,1.0,85.837937,0.92,2.00,3.50,155,29.000000,106.000000,628.885209,302.285704,3.100000,NaN,NaN,NaN,NaN,155,54.298030,57.537141,1.961290,5,304.0,3,2,1,18,40.06,within_video_activity_percentile_not_ml,99.36,OK
3,4,0.0,236.0,237.0,9.44,1.0,1.000000,236.0,0.0,640.232170,9.44,67.821205,128.725000,76.663775,578.751148,408.963967,97.857331,2.0,218.563040,2.08,1.00,2.00,237,91.400000,137.800000,396.496179,230.985100,1.896000,57.0,2.28,12.789,0.7609,237,62.051930,53.693175,1.198312,5,284.0,3,2,1,18,74.01,within_video_activity_percentile_not_ml,100.00,OK
4,5,0.0,56.0,57.0,2.24,1.0,1.000000,56.0,0.0,48.560925,2.24,21.678984,59.097346,14.679891,416.261937,368.521573,37.150549,0.0,0.000000,0.00,1.00,3.00,57,0.000000,56.000000,560.377167,251.346386,2.280000,NaN,NaN,NaN,NaN,57,62.092256,66.598549,1.649123,3,93.0,3,0,1,18,26.40,within_video_activity_percentile_not_ml,100.00,OK
5,6,0.0,121.0,121.0,4.84,1.0,0.991803,120.0,0.0,395.758023,4.84,81.768187,150.787798,92.235754,516.501980,775.539692,104.905540,1.0,82.493609,0.72,1.25,3.25,121,61.250000,90.750000,599.396387,276.936914,1.210000,7.0,0.28,10.301,0.8036,121,37.909352,33.481515,2.776860,6,331.0,1,5,1,18,65.41,within_video_activity_percentile_not_ml,99.18,OK
6,7,0.0,48.0,49.0,1.92,1.0,1.000000,48.0,0.0,57.552918,1.92,29.975478,92.262500,25.434973,250.337591,383.329529,53.968302,1.0,33.424574,0.48,1.00,4.00,49,0.000000,48.000000,724.299367,249.456469,1.960000,NaN,NaN,NaN,NaN,49,109.485386,109.039955,0.000000,0,0.0,0,0,2,1,31.34,within_video_activity_percentile_not_ml,100.00,OK
7,8,0.0,178.0,178.0,7.12,1.0,0.994413,176.0,1.0,357.750636,7.08,50.529751,139.601050,48.346408,850.938096,542.718095,72.570910,2.0,119.279171,1.44,1.00,3.00,178,44.333333,114.333333,558.192232,209.226721,2.373333,NaN,NaN,NaN,NaN,178,50.581255,50.118254,1.370787,5,241.0,2,3,1,18,54.06,within_video_activ


========== LABEL TEMPLATE ==========
Shape: (76, 6)


,canonical_id,team_label,reviewed_player_identity,reviewed_performance_score_0_100,label_source,reviewer_notes
0,1,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN,NaN
2,3,NaN,NaN,NaN,NaN,NaN
3,4,NaN,NaN,NaN,NaN,NaN
4,5,NaN,NaN,NaN,NaN,NaN


In [3]:
feature_quality = pd.DataFrame({
    "data_type": features.dtypes.astype(str),
    "missing_count": features.isna().sum(),
    "missing_percentage": (
        features.isna().mean() * 100
    ).round(2),
    "unique_values": features.nunique(dropna=True),
})

feature_quality["constant_feature"] = (
    feature_quality["unique_values"] <= 1
)

feature_quality = feature_quality.sort_values(
    by=["missing_percentage", "unique_values"],
    ascending=[False, True]
)

display(feature_quality)

quality_path = OUTPUT_DIR / "feature_quality_audit.csv"
feature_quality.to_csv(quality_path)

print("Saved:", quality_path)

,data_type,missing_count,missing_percentage,unique_values,constant_feature
possession_estimated_possession_frames,float64,56,73.68,15,False
possession_estimated_possession_seconds,float64,56,73.68,15,False
possession_mean_ball_distance_px,float64,56,73.68,20,False
possession_mean_heuristic_confidence,float64,56,73.68,20,False
movement_experimental_sprint_duration_seconds,float64,16,21.05,31,False
movement_maximum_positive_acceleration_pixels_per_second_squared,float64,16,21.05,60,False
movement_maximum_deceleration_magnitude_pixels_per_second_squared,float64,16,21.05,60,False
movement_experimental_sprint_threshold_pixels_per_second,float64,16,21.05,60,False
movement_average_speed_pixels_per_second,float64,7,9.21,64,False
movement_maximum_step_speed_pixels_per_second,float64,7,9.21,64,False


Saved: /mnt/d/My Projects/AthletiQ/AthletiQ/outputs/performance/feature_quality_audit.csv


In [4]:
# Diagnose tracks with missing possession data

possession_columns = [
    "possession_estimated_possession_frames",
    "possession_estimated_possession_seconds",
    "possession_mean_ball_distance_px",
    "possession_mean_heuristic_confidence",
]

available_columns = [
    column
    for column in possession_columns
    if column in features.columns
]

missing_possession = features[
    features[available_columns].isna().any(axis=1)
].copy()

print("Tracks with missing possession:", len(missing_possession))
print("Tracks with possession data:", len(features) - len(missing_possession))

diagnostic_columns = [
    "canonical_id",
    "movement_observed_rows",
    "movement_coverage_ratio",
    "spatial_observed_frames",
    "kdtree_observed_frames",
    *available_columns,
]

diagnostic_columns = [
    column
    for column in diagnostic_columns
    if column in features.columns
]

display(
    missing_possession[diagnostic_columns]
    .sort_values("movement_observed_rows", ascending=True)
)

missing_possession[
    diagnostic_columns
].to_csv(
    OUTPUT_DIR / "missing_possession_diagnostic.csv",
    index=False,
)

print(
    "Saved:",
    OUTPUT_DIR / "missing_possession_diagnostic.csv",
)

Tracks with missing possession: 56
Tracks with possession data: 20


,canonical_id,movement_observed_rows,movement_coverage_ratio,spatial_observed_frames,kdtree_observed_frames,possession_estimated_possession_frames,possession_estimated_possession_seconds,possession_mean_ball_distance_px,possession_mean_heuristic_confidence
17,18,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
24,25,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
47,48,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
58,59,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
69,70,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
67,68,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
61,62,1.0,1.000000,1,1,NaN,NaN,NaN,NaN
14,15,2.0,1.000000,2,2,NaN,NaN,NaN,NaN
22,23,2.0,1.000000,2,2,NaN,NaN,NaN,NaN
60,61,2.0,1.000000,2,2,NaN,NaN,NaN,NaN


Saved: /mnt/d/My Projects/AthletiQ/AthletiQ/outputs/performance/missing_possession_diagnostic.csv


In [5]:
# Diagnose whether missing possession values originate
# in the possession output or during the DSA merge.

POSSESSION_PATH = (
    PROJECT_ROOT
    / "outputs/ball_possession/football2_player_possession_summary.csv"
)

if not POSSESSION_PATH.exists():
    print("Original possession summary not found:", POSSESSION_PATH)
else:
    possession = pd.read_csv(POSSESSION_PATH)

    print("Original possession summary shape:", possession.shape)
    print("Original columns:")
    print(possession.columns.tolist())

    display(possession.head(10))

    print("\nRows per canonical ID in original possession summary:")

    if "canonical_id" in possession.columns:
        print(
            "Unique IDs:",
            possession["canonical_id"].nunique()
        )
        print(
            "Duplicate IDs:",
            possession["canonical_id"].duplicated().sum()
        )

        possession_ids = set(
            possession["canonical_id"].dropna()
        )
        feature_ids = set(
            features["canonical_id"].dropna()
        )

        print("IDs in DSA table:", len(feature_ids))
        print("IDs in possession summary:", len(possession_ids))
        print("IDs absent from possession summary:", len(feature_ids - possession_ids))
        print("IDs absent from DSA table:", len(possession_ids - feature_ids))

        print("\nIDs absent from possession summary:")
        print(sorted(feature_ids - possession_ids))

        print("\nIDs present in possession summary but absent from DSA:")
        print(sorted(possession_ids - feature_ids))
    else:
        print(
            "No canonical_id column found. Inspect the column names "
            "before comparing records."
        )

Original possession summary shape: (20, 6)
Original columns:
['canonical_id', 'team', 'estimated_possession_frames', 'estimated_possession_seconds', 'mean_ball_distance_px', 'mean_heuristic_confidence']


,canonical_id,team,estimated_possession_frames,estimated_possession_seconds,mean_ball_distance_px,mean_heuristic_confidence
0,4,team_b,57,2.28,12.789,0.7609
1,24,team_b,57,2.28,42.171,0.5064
2,21,team_b,55,2.20,19.233,0.7281
3,31,team_b,44,1.76,11.403,0.7650
4,13,team_b,36,1.44,19.391,0.6963
5,37,team_a,30,1.20,8.103,0.7122
6,42,team_b,28,1.12,28.578,0.6021
7,41,team_a,28,1.12,30.515,0.5471
8,45,team_b,18,0.72,25.907,0.5758
9,17,team_b,13,0.52,31.840,0.5923



Rows per canonical ID in original possession summary:
Unique IDs: 20
Duplicate IDs: 0
IDs in DSA table: 76
IDs in possession summary: 20
IDs absent from possession summary: 56
IDs absent from DSA table: 0

IDs absent from possession summary:
[1, 2, 3, 5, 7, 8, 9, 10, 11, 15, 16, 18, 19, 20, 22, 23, 25, 26, 27, 28, 29, 32, 34, 35, 36, 38, 39, 43, 44, 46, 48, 49, 51, 52, 53, 54, 55, 56, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76]

IDs present in possession summary but absent from DSA:
[]
